In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
import sys

PROJECT_ROOT = Path().resolve().parents[0]  # notebooks/ is one level down
sys.path.append(str(PROJECT_ROOT / "src"))

In [3]:
from chatGnT.config import CFG, ensure_dirs
import chatGnT.utils as utils
import chatGnT.data.load as load
import chatGnT.data.preprocess as preprocess
import chatGnT.data.tokenize as tokenize
ensure_dirs(CFG)

/Users/slacksa/miniconda3/envs/chatGnT/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
data = load.load_all()
df = data["ingred"]


In [5]:
#TODO: revisit and handle things like "Twist of  Orange peel" "Top with..."
#TODO: how make units consistent - good opportunity for test functions?
df_clean = preprocess.clean_ingred(df)
df_clean.head()


,id,ingredient_name,ingredient_link,ingredient_image,amt,unit,ingred
0,1,1 oz Coconut rum,/ingredient/135-Coconut-rum,../images/ingredients/Coconut rum-Medium.png,1,oz,Coconut rum
1,1,1/2 oz Amaretto,/ingredient/18-Amaretto,../images/ingredients/Amaretto-Medium.png,1/2,oz,Amaretto
2,1,4 oz Orange juice,/ingredient/352-Orange-juice,../images/ingredients/Orange juice-Medium.png,4,oz,Orange juice
3,1,1/2 oz Grenadine,/ingredient/250-Grenadine,../images/ingredients/Grenadine-Medium.png,1/2,oz,Grenadine
4,2,2 oz Light rum,/ingredient/305-Light-rum,../images/ingredients/Light rum-Medium.png,2,oz,Light rum


In [6]:
tokens = tokenize.recipe_to_tokens(df_clean)
print(tokens[0])

['<amt>1</amt>', '<unit>oz</unit>', '<ingred>Coconut rum</ingred>', '<sep>', '<amt>1/2</amt>', '<unit>oz</unit>', '<ingred>Amaretto</ingred>', '<sep>', '<amt>4</amt>', '<unit>oz</unit>', '<ingred>Orange juice</ingred>', '<sep>', '<amt>1/2</amt>', '<unit>oz</unit>', '<ingred>Grenadine</ingred>', '<sep>']


In [7]:
# Flatten all tokens into a single list
all_tokens = [token for recipe in tokens for token in recipe]
# Get unique tokens and assign an ID to each
vocab = {token: i+1 for i, token in enumerate(sorted(set(all_tokens)))}
# See first few entries in the vocabulary
# print({k: vocab[k] for k in list(vocab)[:10]})
# Optional: add a padding token for batching, add a recipe end token
vocab["<pad>"] = 0
vocab["<end>"] = len(vocab)
# Inverse vocab if you want to convert back later
inv_vocab = {i: token for token, i in vocab.items()}

# Check length of each entry in vocab
token_lengths = {token: len(token) for token in vocab}
print(token_lengths)


{'<amt>0.25</amt>': 15, '<amt>0.5</amt>': 14, '<amt>0.75</amt>': 15, '<amt>1 1/2</amt>': 16, '<amt>1 1/3</amt>': 16, '<amt>1 1/4</amt>': 16, '<amt>1 2/3</amt>': 16, '<amt>1 3/4</amt>': 16, '<amt>1.5</amt>': 14, '<amt>1/2</amt>': 14, '<amt>1/3</amt>': 14, '<amt>1/4</amt>': 14, '<amt>1/6</amt>': 14, '<amt>1/8</amt>': 14, '<amt>100</amt>': 14, '<amt>10</amt>': 13, '<amt>125</amt>': 14, '<amt>12</amt>': 13, '<amt>13</amt>': 13, '<amt>150</amt>': 14, '<amt>151</amt>': 14, '<amt>15</amt>': 13, '<amt>16</amt>': 13, '<amt>1750</amt>': 15, '<amt>1</amt>': 12, '<amt>2 1/2</amt>': 16, '<amt>2.5</amt>': 14, '<amt>2/3</amt>': 14, '<amt>200</amt>': 14, '<amt>20</amt>': 13, '<amt>25</amt>': 13, '<amt>28</amt>': 13, '<amt>2</amt>': 12, '<amt>3.5</amt>': 14, '<amt>3/4</amt>': 14, '<amt>30</amt>': 13, '<amt>355</amt>': 14, '<amt>3</amt>': 12, '<amt>4.5</amt>': 14, '<amt>40</amt>': 13, '<amt>45</amt>': 13, '<amt>46</amt>': 13, '<amt>4</amt>': 12, '<amt>50</amt>': 13, '<amt>5</amt>': 12, '<amt>60</amt>': 

In [9]:
# Now, encode tokens as integers
tokens_encoded = [[vocab[t] for t in recipe] for recipe in tokens]
# Example: print the first encoded recipe
# print(tokens_encoded[0])

lengths = [len(r) for r in tokens_encoded]
# print("Min:", min(lengths))
# print("Max:", max(lengths))
# print("Example lengths:", lengths[:10])

# Now add end token and pad to same length (seq_length = 48 + 1 for end token)
# batch_size = 621 recipes 
max_len = max(len(r) for r in tokens_encoded)

# Add end token and pad
tokens_padded = [
    r + [vocab["<end>"]] + [vocab["<pad>"]] * (max_len - len(r))
    for r in tokens_encoded
]

# lengths = [len(r) for r in tokens_padded]
# print("Min:", min(lengths))
# print("Max:", max(lengths))
# print("Example lengths:", lengths[:10])


Min: 49
Max: 49
Example lengths: [49, 49, 49, 49, 49, 49, 49, 49, 49, 49]


In [10]:
# Check first recipe encoding
#TODO: okay to end with <sep>, <end> ?
decoded_recipe = [inv_vocab[i] for i in tokens_padded[0]]
print("Decoded back:", decoded_recipe)

Decoded back: ['<amt>1</amt>', '<unit>oz</unit>', '<ingred>Coconut rum</ingred>', '<sep>', '<amt>1/2</amt>', '<unit>oz</unit>', '<ingred>Amaretto</ingred>', '<sep>', '<amt>4</amt>', '<unit>oz</unit>', '<ingred>Orange juice</ingred>', '<sep>', '<amt>1/2</amt>', '<unit>oz</unit>', '<ingred>Grenadine</ingred>', '<sep>', '<end>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>']


In [11]:
import torch
from torch.nn.utils.rnn import pad_sequence
import torch.nn as nn  # for embedding layer

# tokens_tensor = [torch.tensor(r) for r in tokens_padded]
tokens_tensor = torch.tensor(tokens_padded, dtype=torch.long)
print(tokens_tensor.shape)  # torch.Size([621, 48])

# Need to have [seq_len, batch_size] for TransformerEncoder
tokens_tensor = tokens_tensor.transpose(0, 1)
print(tokens_tensor.shape)

torch.Size([621, 49])
torch.Size([49, 621])


In [32]:
# vocab_size = len(vocab)
# embed_size = 16  #TODO: can change this!
# embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embed_size, padding_idx=0)
# embedded = embedding(tokens_tensor_padded)
#NOTE: Model class does this

# TODO: for now, add padding mask but not a causal mask that block seeing future
pad_id = vocab["<pad>"]   # should be 0
# src_key_padding_mask shape = [batch_size, seq_len]
pad_mask = (tokens_tensor == pad_id).transpose(0, 1)
print(pad_mask.shape)   # [batch_size, seq_len]


torch.Size([621, 49])


In [48]:
import chatGnT.models.transformer as transformer
import time
import math

In [30]:
embed_size=16
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = transformer.TransformerModel(
    ntoken=len(vocab),
    ninp=embed_size,
    nhead=4,
    nhid=256,
    nlayers=2).to(device)
#TODO: investigate error? 


/Users/slacksa/repos/chatGnT/src/chatGnT/models/transformer.py:17: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.transformer_encoder = TransformerEncoder(encoder_layers, nlayers)


In [33]:
#TODO: understand these!
learning_rate = 1e-3
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
criterion = torch.nn.CrossEntropyLoss(ignore_index=pad_id)

In [43]:
def train():
    model.train() # Turn on the train mode
    total_loss = 0.
    start_time = time.time()
    
    seq_len = train_src.size(0)
    src_mask = model.generate_square_subsequent_mask(seq_len).to(device)
    # src_mask = model.generate_square_subsequent_mask(bptt).to(device)
    # for batch, i in enumerate(range(0, train_data.size(0) - 1, bptt)):
    # data, targets = get_batch(train_data, i)
    optimizer.zero_grad()
    # if data.size(0) != bptt:
    #     src_mask = model.generate_square_subsequent_mask(data.size(0)).to(device)

    # Forward pass
    output = model(
        src=train_src,
        src_key_padding_mask=train_pad_mask[:, :-1],
        src_mask=src_mask)

    loss = criterion(
        output.view(-1, ntokens),
        train_tgt.reshape(-1))

    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
    optimizer.step()

    total_loss += loss.item()
    # log_interval = 200
    # if batch % log_interval == 0 and batch > 0:
        # cur_loss = total_loss / log_interval
    elapsed = time.time() - start_time
    print('| epoch {:3d} | '
            'lr {:02.2f} | '
            'loss {:5.2f} | ppl {:8.2f}'.format(
            epoch,
            optimizer.param_groups[0]['lr'],
            elapsed,
            loss.item(),
            math.exp(loss.item())))
    # total_loss = 0
    start_time = time.time()

In [51]:
def evaluate(eval_model):
    eval_model.eval() # Turn on the evaluation mode
    # total_loss = 0.

    seq_len = train_src.size(0)
    src_mask = model.generate_square_subsequent_mask(seq_len).to(device)

    with torch.no_grad():
        # for i in range(0, data_source.size(0) - 1, bptt):
            # data, targets = get_batch(data_source, i)
            # if data.size(0) != bptt:
                # src_mask = model.generate_square_subsequent_mask(data.size(0)).to(device)
            output = model(
                src=train_src,
                src_key_padding_mask=train_pad_mask[:, :-1],
                src_mask=src_mask)
            output_flat = output.view(-1, ntokens)
            # total_loss += len(data) * criterion(output_flat, targets).item()
            loss = criterion(
                output.reshape(-1, ntokens),
                val_tgt.reshape(-1))
    # return total_loss / (len(data_source) - 1)
    return loss.item()

In [56]:
best_val_loss = float("inf")
epochs = 3 # The number of epochs
best_model = None
ntokens = len(vocab)

# For training
train_src = tokens_tensor[:-1, :]       # all tokens except last
train_tgt = tokens_tensor[1:, :]        # shifted by one
train_pad_mask = (tokens_tensor == pad_id).transpose(0, 1)

# For validation, if you have a separate val_tokens_tensor
#TODO: eventually update so not same as training
val_src = tokens_tensor[:-1, :]
val_tgt = tokens_tensor[1:, :]
val_pad_mask = (tokens_tensor == pad_id).transpose(0, 1)

for epoch in range(1, epochs + 1):
    epoch_start_time = time.time()

    train_loss = train()
    val_loss = evaluate(model)

    print('-' * 89)
    print('| end of epoch {:3d} | time: {:5.2f}s | '
          'valid loss {:5.2f} | valid ppl {:8.2f}'.format(
              epoch,
              (time.time() - epoch_start_time),
              val_loss,
              math.exp(val_loss)))
    print('-' * 89)


| epoch   1 | lr 0.00 | loss  0.23 | ppl     6.29
-----------------------------------------------------------------------------------------
| end of epoch   1 | time:  0.26s | valid loss  6.22 | valid ppl   504.47
-----------------------------------------------------------------------------------------
| epoch   2 | lr 0.00 | loss  0.24 | ppl     6.26
-----------------------------------------------------------------------------------------
| end of epoch   2 | time:  0.27s | valid loss  6.19 | valid ppl   489.87
-----------------------------------------------------------------------------------------
| epoch   3 | lr 0.00 | loss  0.24 | ppl     6.23
-----------------------------------------------------------------------------------------
| end of epoch   3 | time:  0.28s | valid loss  6.17 | valid ppl   476.41
-----------------------------------------------------------------------------------------
